<a href="https://colab.research.google.com/github/AntonioLLuis/Algoritmo-e-extrutura-de-dados/blob/main/Biblioteca_POO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# O arquivo mantém a mesma lógica funcional do original, mas organiza tudo em uma
# classe Biblioteca que encapsula estado e comportamento.

# IMPORTS
import json  # usado para serializar/deserializar os dados para/de JSON
import os    # usado para operações do sistema de arquivos (verificar existência de arquivo)

# CONSTANTE
ARQUIVO = "biblioteca.json"  # nome do arquivo onde os dados são persistidos (mesmo do procedural)


# ------------------------------------------------------------
# EXPLICAÇÃO GERAL DA CONVERSÃO PARA POO (RESUMO ANTES DO CÓDIGO)
# ------------------------------------------------------------
# Por que POO aqui?
# - Encapsulamento: todas as funções e dados relacionados à biblioteca ficam dentro de
#   uma única classe (Biblioteca). Isso reduz variáveis "soltas" e facilita manutenção.
# - Reusabilidade: criar múltiplas instâncias permite, por exemplo, múltiplos catálogos
#   em memória sem conflitos (útil para testes ou cenários multi-usuário).
# - Testabilidade: métodos isolados são fáceis de testar unitariamente (ex.: testar emprestar).
# - Organização: separar responsabilidades em métodos torna o código mais legível e extensível.
#
# Observação: Para scripts muito pequenos e pontuais, POO pode parecer verbosa. Use POO quando
# espera evolução, testes automáticos, ou quando quiser separar lógica da apresentação.

class Biblioteca:
    """
    Classe que representa o sistema de biblioteca.
    Em vez de funções soltas, encapsulamos o estado (lista de livros) e métodos
    que operam sobre esse estado (cadastrar, listar, emprestar, etc).
    """

    def __init__(self, arquivo=ARQUIVO):
        """
        Construtor da classe.
        - arquivo: nome do arquivo JSON para persistência.
        Explicação:
        - Em procedural, ARQUIVO era uma constante global usada pelas funções.
          Aqui guardamos esse nome como atributo da instância para permitir variações
          (ex: em testes usar um arquivo temporário).
        """
        # armazenamos o caminho do arquivo como atributo da instância
        self.arquivo = arquivo

        # atributo que conterá a lista de livros em memória (cada livro é um dict)
        self.livros = []

        # ao criar a instância, carregamos os livros do arquivo (se existir)
        # assim a instância já começa pronta para operar.
        self.carregar_livros()

    # ----------------------------
    # MÉTODOS DE PERSISTÊNCIA
    # ----------------------------
    def carregar_livros(self):
        """
        Carrega os livros do arquivo JSON definido em self.arquivo.
        - Se o arquivo não existe, inicializa self.livros como lista vazia.
        Explicação linha a linha:
        - usamos os.path.exists para verificar existência do arquivo (evita exceção).
        - usamos json.load para transformar o JSON em estrutura Python (lista/dicts).
        """
        # checa se o arquivo existe; retorna True/False
        if not os.path.exists(self.arquivo):
            # se não existe, garantimos que o atributo seja uma lista vazia
            self.livros = []
            return  # fim do método: nada a ler

        # abre o arquivo para leitura usando with para fechar automaticamente
        with open(self.arquivo, "r", encoding="utf-8") as f:
            # carrega o JSON do arquivo e armazena em self.livros
            # json.load converte JSON em listas/dicionários Python
            self.livros = json.load(f)

    def salvar_livros(self):
        """
        Salva self.livros no arquivo JSON (self.arquivo).
        - Usa json.dump com indent=4 e ensure_ascii=False para legibilidade e suporte a acentos.
        """
        # abre (ou cria) o arquivo no modo escrita; sobrescreve conteúdo anterior
        with open(self.arquivo, "w", encoding="utf-8") as f:
            # escreve a lista de livros em formato JSON legível
            json.dump(self.livros, f, indent=4, ensure_ascii=False)

    # ----------------------------
    # MÉTODOS DE AÇÃO (CRUD e negócios)
    # ----------------------------
    def cadastrar_livro(self):
        """
        Cadastra um novo livro solicitando dados ao usuário e persistindo no arquivo.
        - Verifica duplicidade de código antes de inserir.
        """
        print("\n--- CADASTRAR NOVO LIVRO ---")
        # input retorna string; strip remove espaços desnecessários
        codigo = input("Código do livro: ").strip()

        # verifica se já existe um livro com o mesmo código (procura linear)
        for livro in self.livros:
            # compara o campo "codigo" do dicionário com o código informado
            if livro["codigo"] == codigo:
                print("Já existe um livro com esse código.")
                return  # encerra sem cadastrar

        # solicita os demais campos
        titulo = input("Título do livro: ").strip()
        autor = input("Autor: ").strip()
        ano = input("Ano de publicação: ").strip()

        # cria o dicionário representando o livro
        novo_livro = {
            "codigo": codigo,
            "titulo": titulo,
            "autor": autor,
            "ano": ano,
            "status": "disponível"  # status inicial conforme seu script procedural
        }

        # adiciona o novo livro à lista em memória
        self.livros.append(novo_livro)

        # persiste imediatamente a alteração no arquivo
        self.salvar_livros()

        # confirma ao usuário
        print("Livro cadastrado com sucesso!")

    def listar_livros(self):
        """
        Exibe todos os livros cadastrados de forma legível.
        - Se a lista estiver vazia, informa ao usuário.
        """
        print("\n--- LISTA DE LIVROS ---")
        # testa se não há livros cadastrados
        if not self.livros:
            print("Nenhum livro cadastrado.")
            return

        # itera sobre cada livro (dicionário) e imprime os campos relevantes
        for livro in self.livros:
            # f-string com multiline para facilitar leitura
            print(f"""
Código: {livro['codigo']}
Título: {livro['titulo']}
Autor: {livro['autor']}
Ano: {livro['ano']}
Status: {livro['status']}
""")

    def buscar_livro(self):
        """
        Busca livros por título (substring, case-insensitive) ou por código (exato).
        - Solicita termo de busca ao usuário e mostra resultados resumidos.
        """
        print("\n--- BUSCAR LIVRO ---")
        termo = input("Digite o título ou código: ").strip().lower()

        encontrados = []  # lista para acumular resultados

        # percorre os livros procurando correspondência
        for livro in self.livros:
            # converte título do livro para minúsculas e verifica substring
            # compara também código exatamente (código já foi normalizado com strip)
            if termo in livro["titulo"].lower() or termo == livro["codigo"]:
                encontrados.append(livro)

        # se nada encontrado, informa
        if not encontrados:
            print("Nenhum livro encontrado.")
            return

        # caso encontre, mostra quantos e lista resumo
        print(f"\n{len(encontrados)} livro(s) encontrado(s):\n")
        for livro in encontrados:
            print(f"Código: {livro['codigo']} - {livro['titulo']} ({livro['status']})")

    def remover_livro(self):
        """
        Remove um livro por código, se existir.
        - Atualiza o arquivo após remoção.
        """
        print("\n--- REMOVER LIVRO ---")
        codigo = input("Código do livro a remover: ").strip()

        # iteramos sobre uma cópia da lista ou usamos índice; aqui usamos for clássico com remoção e return
        for livro in self.livros:
            if livro["codigo"] == codigo:
                # remove o dicionário da lista
                self.livros.remove(livro)
                # salva imediatamente o estado modificado
                self.salvar_livros()
                print("Livro removido com sucesso!")
                return

        # se não encontrou nenhum com o código informado
        print("Nenhum livro encontrado com esse código.")

    def emprestar_livro(self):
        """
        Marca o livro como 'emprestado' se estiver disponível.
        - Se já estiver emprestado, informa o usuário.
        """
        print("\n--- EMPRESTAR LIVRO ---")
        codigo = input("Código do livro: ").strip()

        for livro in self.livros:
            if livro["codigo"] == codigo:
                # verifica status atual
                if livro["status"] == "emprestado":
                    print("Este livro já está emprestado!")
                    return
                # marca como emprestado e salva
                livro["status"] = "emprestado"
                self.salvar_livros()
                print("Livro emprestado com sucesso!")
                return

        print("Livro não encontrado.")

    def devolver_livro(self):
        """
        Marca o livro como 'disponível' se estiver emprestado.
        - Se já estiver disponível, informa o usuário.
        """
        print("\n--- DEVOLVER LIVRO ---")
        codigo = input("Código do livro: ").strip()

        for livro in self.livros:
            if livro["codigo"] == codigo:
                if livro["status"] == "disponível":
                    print("Este livro já está disponível!")
                    return
                livro["status"] = "disponível"
                self.salvar_livros()
                print("Livro devolvido com sucesso!")
                return

        print("Livro não encontrado.")

    # ----------------------------
    # MÉTODO DE INTERAÇÃO (MENU)
    # ----------------------------
    def menu(self):
        """
        Apresenta o menu principal e orquestra as operações.
        - Esse método substitui a função menu() do seu script procedural.
        - Mantém o mesmo fluxo de interação com o usuário.
        """
        # loop principal do menu - executa até o usuário escolher sair
        while True:
            # cabeçalho do sistema (mesmo texto que estava no script original)
            print("\n===== SISTEMA DE BIBLIOTECA =====")
            print("1 - Cadastrar livro")
            print("2 - Listar livros")
            print("3 - Buscar livro")
            print("4 - Remover livro")
            print("5 - Emprestar livro")
            print("6 - Devolver livro")
            print("7 - Sair")

            # lê a opção do usuário
            opcao = input("Escolha uma opção: ")

            # roteamento das opções para os métodos da instância
            if opcao == "1":
                # chamamos o método cadastrar_livro, que modifica self.livros e salva
                self.cadastrar_livro()
            elif opcao == "2":
                self.listar_livros()
            elif opcao == "3":
                self.buscar_livro()
            elif opcao == "4":
                self.remover_livro()
            elif opcao == "5":
                self.emprestar_livro()
            elif opcao == "6":
                self.devolver_livro()
            elif opcao == "7":
                print("Saindo do sistema...")
                break  # encerra loop e finaliza menu()
            else:
                # trata entradas inválidas (mesmo comportamento do script original)
                print("Opção inválida! Tente novamente.")


# ----------------------------
# BLOCO DE EXECUÇÃO DIRETA
# ----------------------------
# Mantemos a convenção: se o arquivo for executado diretamente, criamos a instância
# e chamamos biblioteca.menu(). Isso é equivalente ao comportamento do script procedural.
if __name__ == "__main__":
    # criamos uma instância da classe Biblioteca com o arquivo padrão ARQUIVO
    bib = Biblioteca(arquivo=ARQUIVO)
    # iniciamos o menu que fará a interação com o usuário
    bib.menu()


# ------------------------------------------------------------
# ANOTAÇÕES / EXPLICAÇÕES FINAIS (aprender com POO)
# ------------------------------------------------------------
# 1) O que mudou estruturalmente:
#    - Antes: funções soltas que recebiam a lista 'livros' como parâmetro.
#    - Agora: a lista de livros é um atributo da instância (self.livros) e as operações são métodos.
#
# 2) Vantagens práticas desta organização:
#    - Facilidade para testes automatizados: instancie Biblioteca(arquivo="test.json") e chame métodos.
#    - Facilidade para extensão: ex.: adicionar logging, permissões, salvar em DB, criar GUI.
#    - Menos risco de variáveis globais e efeitos colaterais inesperados.
#
# 3) Casos onde procedural é suficiente:
#    - Scripts muito pequenos, usados uma vez, sem intenção de evoluir.
#    - Se preferir simplicidade imediata, o estilo procedural é mais direto.
#
# 4) Observações sobre comentários:
#    - Mantive comentários explicativos próximos ao código, como você pediu.
#    - Para produção, recomenda-se balancear comentários: documentar intenções e complexidade
#      mas evitar comentários redundantes em cada linha quando o código for já claro.
#
# Se quiser eu:
# - removo/compacto comentários para versão "limpa" para execução;
# - adiciono validações extras (ex.: validar ano, impedir códigos vazios, confirmação na remoção);
# - adapto para usar SQLite/CSV/Excel ao invés de JSON;
# - crio testes unitários (pytest) para cada método (ex.: cadastrar, emprestar, devolver).
#
# Diga qual próximo passo prefere — posso também converter outro script para POO
# com o mesmo nível de detalhamento.
